In [ ]:
:< rust_operators

#### optionm (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:= default_value d =
    optionm::defaultWith d

#### default_value (test)

In [1]:
///- --test --symbol

None
|> default_value 3i32
|> _assert_eq 3i32

__assert_eq { expected = 3 }

In [ ]:
:= (/??) a b =
    a |> default_value b

In [2]:
///- --test --symbol

None /?? 3i32
|> _assert_eq 3i32

__assert_eq { expected = 3 }

In [ ]:
:= default_with fn = \|
    | Some x => x
    | None => fn ()

#### default_with (test)

In [3]:
///- --test --symbol

None
|> default_with \ () => 3i32
|> _assert_eq 3i32

__assert_eq { expected = 3 }

In [ ]:
:= choose fn a b =
    ?= a, b ?|
    | Some x, Some y => fn x y |> Some
    | _ => None

#### choose (test)

In [4]:
///- --test --symbol

(Some 2i32, Some 3)
||> choose (+)
|> _assert_eq (Some 5)

__assert_eq { expected = Some(5) }

In [ ]:
:= iter fn = \|
    | Some x => fn x
    | None => ()

#### iter (test)

In [5]:
///- --test --symbol

:= n = mut 1i32
:= fn =
    \ n' =>
        n <- *n + n'
Some 1i32 |> iter fn
None |> iter fn
*n
|> _assert_eq 2i32

__assert_eq { expected = 2 }

In [ ]:
:= flatten x =
    ?= x ?|
    | Some (Some x) => Some x
    | _ => None

:= option'_types () : () =
    backend_switch {
        Gleam = \ () => global "import gleam/option"
    }
:! option'_gleam t =
    `(
        option'_types ()
        $'' : $'option.Option(`t)'
    )
:== option'_switch t =
    {
        Gleam : option'_gleam t
        Lua : t
        Fsharp : $"`t option"
        Python : t
        Rust : $'Option<`t>'
        TypeScript : t
        Zig : $'?`(t)'
        Lean : $'(Option `t)'
    }
:! option' t = $'backend_switch `(option'_switch t)'

:= none' \* t. () : option' t =
    backend_switch {
        Gleam = \ () => $'option.None' : option' t
        Lua = \ () => $'nil' : option' t
        Fsharp = \ () => $'None' : option' t
        Python = \ () => $'None' : option' t
        Rust = \ () => $'None' : option' t
        TypeScript = \ () => $'undefined' : option' t
        Zig = \ () => $'null' : option' t
        Lean = \ () => $'none' : option' t
    }

:= some' \* t. (x : t) : option' t =
    backend_switch {
        Gleam = \ () => $'!x |> option.Some' : option' t
        Lua = \ () => $'!x ' : option' t
        Fsharp = \ () => $'Some !x ' : option' t
        Python = \ () => $'!x ## some\' ' : option' t
        Rust = \ () => $'Some(!x.clone())' : option' t
        TypeScript = \ () => $'!x ' : option' t
        Zig = \ () => $'!x ' : option' t
        Lean = \ () => $'(some !x)' : option' t
    }

:= is_none' \* t. (x : option' t) : bool =
    backend_switch {
        Lua = \ () => $'!x == nil' : bool
        Python = \ () => $'!x is None' : bool
        Rust = \ () => $'!x.is_none()' : bool
        TypeScript = \ () => $'!x === undefined' : bool
    }

:= nullable_value' \* t. (x : option' t) : t =
    $'!x' : t

:= default_value' \* t. (value : t) (x : option' t) : t =
    backend_switch {
        Gleam = \ () => $'!x |> option.unwrap(!value)' : t
        Lua = \ () => ? is_none' x ?> value |: nullable_value' x
        Fsharp = \ () => $'!x |> Option.defaultValue !value ' : t
        Python = \ () => ? is_none' x ?> value |: nullable_value' x
        Rust = \ () =>
            := value = rust::clone value
            $'!x.unwrap_or(!value)' : t
        TypeScript = \ () => ? is_none' x ?> value |: nullable_value' x
        Zig = \ () => $'(!x orelse !value)' : t
        Lean = \ () => $'((!x).getD !value)' : t
    }

:= get' \* t. (x : option' t) : t =
    backend_switch {
        Gleam = \ () =>
            option'_types ()
            $'!x |> option.unwrap (Nil)' : t
        Lua = \ () => $'!x ' : t
        Fsharp = \ () => $'!x |> Option.get' : t
        Python = \ () => $'!x ' : t
        Rust = \ () => $'!x.unwrap()' : t
        TypeScript = \ () => $'!x ' : t
        Zig = \ () => $'!x.?' : t
        Lean = \ () => $'(!x).get\!' : t
    }

:= box \* t. (x : option t) : option' t =
    ?= x ?|
    | Some x => some' x
    | None => none' ()

:= map \* t u. (fn : t -> u) (x : option' t) : option' u =
    backend_switch {
        Gleam = \ () =>
            option'_types ()
            $'!x |> option.map(!fn)' : option' u
        Lua = \ () =>
            ? x =. none' ()
            ?> none' ()
            |: fn $'!x ' |> \ x => $'!x ' : option' u
        Fsharp = \ () =>
            := fn = ~~ fn
            $'!x |> Option.map !fn ' : option' u
        Python = \ () =>
            ? x =. none' ()
            ?> none' ()
            |: fn $'!x ' |> \ x => $'!x ' : option' u
        Rust = \ () =>
            := fn = rust::one_param_fn fn
            := x = ~~ x
            $'!x.map(|x| !fn(x))' : option' u
        TypeScript = \ () =>
            ? $'!x === undefined' : bool
            ?> none' ()
            |: fn $'!x ' |> \ x => $'!x ' : option' u
        Zig = \ () =>
            ? $'(!x == null)' : bool
            ?> none' ()
            |: x |> get' |> fn |> some'
        Lean = \ () =>
            ? $'(!x).isNone' : bool
            ?> none' ()
            |: x |> get' |> fn |> some'
    }

:= map'' \* t u. (fn : t -> u) (x : option' t) : option' u =
    x |> map fn

:= unbox \* t. (x : option' t) : option t =
    x |> map'' Some |> default_value' None

#### unbox (test)

In [6]:
///- --test --symbol
///> gleam
///> cuda
///> rust
///> typescript

:= x = Some 3i32
:= y : option i32 = None
:= x' = x |> box |> unbox
:= y' = y |> box |> map id |> unbox
(x', y') |> _assert_eq' (x, y)

.gleam (gleam)
__assert_eq' { expected = (Some(3), None) }
.py (cuda)
__assert_eq' { expected = (Some(3), None) }
.rs (rust)
__assert_eq' { expected = (Some(3), None) }
.ts (typescript)
__assert_eq' { expected = (Some(3), None) }

0

In [ ]:
:= of_obj \* t. (x : t) : option' t =
    backend_switch {
        Gleam = \ () =>
            option'_types ()
            $'!x |> option.Some' : option' t
        Lua = \ () =>
            $'!x ' : option' t
        Fsharp = \ () => (x |> $'Option.ofObj') : option' t
        Python = \ () =>
            $'!x ' : option' t
        Rust = \ () => $'Some(!x.clone())' : option' t
        TypeScript = \ () => $'(!x ?? undefined)' : option' t
        Zig = \ () => $'!x ' : option' t
    }

#### of_obj (test)

In [7]:
///- --test --symbol
///> fsharp
///> cuda
///> typescript
///> cuda

null ()
|> of_obj
|> unbox
|> _assert_eq (None : option string)

.fsx (fsharp)
__assert_eq { expected = None }
.py (cuda)
__assert_eq { expected = None }
.ts (typescript)
__assert_eq { expected = None }
.py (cuda)
__assert_eq { expected = None }

In [8]:
///- --test --symbol
///> gleam
///> cuda
///> rust
///> typescript

""
|> of_obj
|> unbox
|> _assert_eq' (Some "")

.gleam (gleam)
__assert_eq' { expected = Some() }
.py (cuda)
__assert_eq' { expected = Some() }
.rs (rust)
__assert_eq' { expected = Some() }
.ts (typescript)
__assert_eq' { expected = Some() }

0

In [9]:
:= try' \* t. (x : option' t) : t =
    !\\(x, $'"$0?"')

:= map' \* t u. (fn : t -> u) (x : option' t) : option' u =
    (!\\(x, $'"true; let _optionm_map_ = $0.map(|x| { //"') : bool) |> ignore
    := result = fn !\($'"x"')
    (!\\(result, $'"true; $0 })"') : bool) |> ignore
    !\($'"_optionm_map_"')

:= unwrap \* t. (x : option' t) : t =
    !\\(x, $'"$0.unwrap()"')

:= take \* t. (x : option' t) : option' t =
    (!\\(x, $'"true; let mut !x = !x"') : bool) |> ignore
    !\\(x, $'"Option::take(&mut $0)"')

:= take_ref \* t. (x : rust::ref (option' t)) : option' t =
    (!\\(x, $'"true; let mut !x = !x"') : bool) |> ignore
    !\\(x, $'"Option::take(&mut $0)"')

:= take_ref_mut \* t. (x : rust::ref (rust::mut' (option' t))) : option' t =
    !\\(x, $'"Option::take($0)"')

:= cloned \* t. (x : option' (rust::ref t)) : option' t =
    !\\(x, $'"$0.cloned()"')

:= as_ref \* t. (x : rust::ref (option' t)) : option' (rust::ref t) =
    !\\(x, $'"$0.as_ref()"')

:= as_mut \* t. (x : rust::ref (rust::mut' (option' t))) : option' (rust::ref (rust::mut' t)) =
    !\\(x, $'"$0.as_mut()"')

:= unwrap_or \* t. (def : t) (x : option' t) : t =
    !\($'"!x.unwrap_or(!def)"')

:= and_then \* t u. (fn : t -> option' u) (x : option' t) : option' u =
    !\\((x, fn), $'"$0.and_then(|x| $1(x))"')

:= rc_upgrade \* t. (x : rust::weak_rc t) : option' (rust::rc t) =
    !\\(x, $'"std::rc::Weak::upgrade(&$0)"')

:= rc_into_inner \* t. (x : rust::rc t) : option' t =
    !\\(x, $'"std::rc::Rc::into_inner($0)"')

#### rc_into_inner (test)

In [10]:
///- --test --symbol
///> rust

rust::new_rc 0i32
|> rc_into_inner
|> unbox
|> _assert_eq' (Some 0i32)

__assert_eq' { expected = Some(0) }

0